In [ ]:
# ============================================================
# 09_llm_judge_crossmodel.ipynb
# LLM-as-a-Judge quality evaluation (ongoing model-monitoring
# control) with three expert personas and seven dimensions, plus
# cross-model validation using a second, independent judge model.
#
# REQUIRES an OpenAI API key at runtime (.env; see .env.example).
# The key is never hard-coded or committed. This notebook is not
# executed in the offline reproduction run.
#
# Input : data/processed/llm_guides.json  (from nb 08)
# Output: data/processed/judge_scores_raw.csv,
#         results/tables/judge_scores_by_persona_dimension.csv,
#         results/tables/cross_model_correlation.csv,
#         results/figures/judge_scores_heatmap.(png|pdf),
#         results/figures/cross_model_agreement.(png|pdf)
#
# Judges: primary = gpt-4o-mini (temperature 0.0, deterministic),
#         independent = gpt-4o (temperature 0.0). Cross-model
#         Pearson r assesses whether results depend on the judge.
# Figures grayscale, dpi 600, PNG + PDF, no captions.
# ============================================================

import os
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from openai import OpenAI
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
PRIMARY_JUDGE = "gpt-4o-mini"
INDEPENDENT_JUDGE = "gpt-4o"

sns.set_theme(style="white", context="paper")
mpl.rcParams.update({"figure.dpi": 120, "savefig.dpi": 600, "savefig.bbox": "tight",
                     "font.family": "serif", "font.size": 10, "image.cmap": "Greys"})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

DIMENSIONS = {
    "D1_medical_accuracy": "Medical Accuracy",
    "D2_association_distinction": "Association/Causation Distinction",
    "D3_actionability": "Actionability",
    "D4_readability": "Readability & Clarity",
    "D5_disclaimer_adequacy": "Disclaimer Adequacy",
    "D6_insurance_relevance": "Insurance Relevance",
    "D7_overall_quality": "Overall Quality",
}
SCORE_RUBRIC = ("Score each dimension on a 1-5 scale (1 = poor, 3 = acceptable, "
                "5 = excellent). Be strict and consistent with your persona's remit.")

PERSONA_PROMPTS = {
    "medical_expert": """You are an experienced medical professional (internal \
medicine specialist with expertise in diabetes and metabolic disorders) \
evaluating AI-generated health risk consultation guides for quality assurance.

Your focus areas:
- Clinical accuracy of the health information provided
- Whether statistical associations are clearly distinguished from causal claims
- Safety: does the content avoid giving direct medical diagnoses or prescriptions?
- Whether the limitations of cross-sectional data are adequately communicated
- Appropriateness of lifestyle modification suggestions based on evidence
You are NOT evaluating grammar or formatting -- focus on medical substance.""",
    "insurance_specialist": """You are a senior actuary and health insurance \
underwriting specialist evaluating AI-generated health risk consultation guides \
for potential use in insurance risk communication.

Your focus areas:
- Relevance of the risk information for insurance risk stratification
- Whether the probabilistic risk framing is actuarially sound
- Compliance considerations: does the content avoid discriminatory framing?
- Whether the guide could support personalised insurance product design
- Whether appropriate disclaimers are present to limit legal liability
You are NOT evaluating clinical accuracy -- focus on insurance/business utility.""",
    "general_public": """You are a member of the general public (40-year-old \
office worker with no medical or statistical background) reading an AI-generated \
health risk consultation guide about your diabetes risk.

Your focus areas:
- Can you understand the main message without medical knowledge?
- Are the recommended actions clear and realistic in daily life?
- Does it feel helpful and empathetic rather than alarming?
- Is it clear that this is NOT a medical diagnosis?
- Would you know what to do next after reading this?
You are NOT evaluating technical accuracy -- focus on everyday usability.""",
}

def build_judge_prompt(case, persona_name):
    dim_lines = "\n".join(f"  {i+1}. {v} ({k})" for i, (k, v) in enumerate(DIMENSIONS.items()))
    return f"""
=== EVALUATION TASK ===

You are acting as: {persona_name.replace('_', ' ').upper()}

Below is an AI-generated health risk consultation guide produced for a \
{case['gender'].capitalize()}, age {case['age']}, with a statistical diabetes \
risk probability of {case['original_prob']:.1%} (counterfactual scenario: \
{case['cf_prob']:.1%}, reduction: {case['prob_reduction']:.1%} points), generated \
from counterfactual explanations of an XGBoost model trained on cross-sectional \
survey data (KNHANES 2020-2024).

=== CONSULTATION GUIDE TO EVALUATE ===

{case['llm_output']}

=== EVALUATION INSTRUCTIONS ===

{SCORE_RUBRIC}

Evaluate the guide on ALL 7 dimensions below:
{dim_lines}

RESPONSE FORMAT (JSON only -- no preamble, no markdown fences):
{{
  "D1_medical_accuracy": <int 1-5>,
  "D2_association_distinction": <int 1-5>,
  "D3_actionability": <int 1-5>,
  "D4_readability": <int 1-5>,
  "D5_disclaimer_adequacy": <int 1-5>,
  "D6_insurance_relevance": <int 1-5>,
  "D7_overall_quality": <int 1-5>,
  "strengths": "<1-2 sentences>",
  "weaknesses": "<1-2 sentences>",
  "persona_specific_note": "<1 sentence>"
}}
Return ONLY the JSON object.""".strip()

def call_judge(case, persona_name, model, temperature=0.0, retry=3, wait=5):
    system = PERSONA_PROMPTS[persona_name]
    user = build_judge_prompt(case, persona_name)
    for attempt in range(retry):
        try:
            resp = client.chat.completions.create(
                model=model, temperature=temperature,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}])
            parsed = json.loads(resp.choices[0].message.content)
            for k in DIMENSIONS:
                parsed[k] = int(max(1, min(5, int(parsed[k]))))
            parsed.update(persona=persona_name, case_id=case["case_id"], judge_model=model)
            return parsed
        except Exception as e:
            print(f"  [attempt {attempt+1}/{retry}] {persona_name}/{model}: {e}")
            time.sleep(wait)
    return None

# ---- Run both judge models over all guides x personas ----
guides = json.load(open(PROC / "llm_guides.json", encoding="utf-8"))
PERSONAS = list(PERSONA_PROMPTS.keys())
rows = []
for model in (PRIMARY_JUDGE, INDEPENDENT_JUDGE):
    print(f"Judging with {model}: {len(guides)} guides x {len(PERSONAS)} personas")
    for case in guides:
        for persona in PERSONAS:
            r = call_judge(case, persona, model)
            if r:
                rows.append(r)

raw = pd.DataFrame(rows)
raw.to_csv(PROC / "judge_scores_raw.csv", index=False)

# ---- Aggregate: mean by persona x dimension (primary judge) ----
dim_cols = list(DIMENSIONS.keys())
prim = raw[raw["judge_model"] == PRIMARY_JUDGE]
agg = prim.groupby("persona")[dim_cols].mean().round(2)
agg.to_csv(TAB / "judge_scores_by_persona_dimension.csv")
print("\nPrimary-judge mean scores (persona x dimension):\n", agg.to_string())

# ---- Cross-model correlation per dimension ----
corr_rows = []
for k in dim_cols:
    a = raw[raw["judge_model"] == PRIMARY_JUDGE].set_index(["case_id", "persona"])[k]
    b = raw[raw["judge_model"] == INDEPENDENT_JUDGE].set_index(["case_id", "persona"])[k]
    j = pd.concat([a, b], axis=1, keys=["primary", "independent"]).dropna()
    if j["primary"].std() == 0 or j["independent"].std() == 0:
        r, agree = np.nan, ("Perfect" if (j["primary"] == j["independent"]).all() else "Constant")
    else:
        r = stats.pearsonr(j["primary"], j["independent"])[0]
        agree = "Strong" if r >= .7 else "Moderate" if r >= .4 else "Weak"
    corr_rows.append(dict(Dimension=DIMENSIONS[k], Pearson_r=None if np.isnan(r) else round(r, 3),
                          Primary_mean=round(j["primary"].mean(), 2),
                          Independent_mean=round(j["independent"].mean(), 2), Agreement=agree))
corr = pd.DataFrame(corr_rows)
corr.to_csv(TAB / "cross_model_correlation.csv", index=False)
print("\nCross-model correlation:\n", corr.to_string(index=False))

# ---- Figure: persona x dimension score heatmap (grayscale) ----
fig, ax = plt.subplots(figsize=(8.5, 3.2))
sns.heatmap(agg[dim_cols], annot=True, fmt=".2f", cmap="Greys", vmin=1, vmax=5,
            cbar_kws={"label": "Mean score"}, linewidths=0.4, linecolor="0.8",
            xticklabels=[DIMENSIONS[c].split(" (")[0] for c in dim_cols], ax=ax)
ax.set_xlabel(""); ax.set_ylabel("")
plt.xticks(rotation=30, ha="right")
save_fig(fig, "judge_scores_heatmap")

# ---- Figure: cross-model Pearson r by dimension (grayscale bars) ----
cc = corr.dropna(subset=["Pearson_r"])
fig, ax = plt.subplots(figsize=(6.0, 3.4))
ax.bar(range(len(cc)), cc["Pearson_r"], color="#8C8C8C", edgecolor="black")
ax.axhline(0.7, color="black", lw=1.0, linestyle="--")
ax.set_xticks(range(len(cc)))
ax.set_xticklabels(cc["Dimension"].str.replace(r"\s*\(.*\)", "", regex=True), rotation=30, ha="right")
ax.set_ylabel("Cross-model Pearson $r$"); ax.set_ylim(0, 1)
save_fig(fig, "cross_model_agreement")
print("Saved judge scores, cross-model correlation, and figures.")
